# Demo del capitolo 1 — audiolab

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
ASSET_BASE = None
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-f0eecc423f2de74c")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
        url = ASSET_BASE + "/audiolab-course.zip?v=f0eecc423f2de74c"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
    if ASSET_BASE is None:
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        ASSET_BASE = str(location.origin) + prefisso + "/_jupyterlite/files"
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 44100
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


## Demo 1 — Le particelle oscillano, la perturbazione si sposta

In [ ]:
frame = 0.25                 # frazione di periodo
lunghezza_onda = 1.8          # metri
spostamento = 0.075           # metri
al.longitudinal_wave(frame=frame, wavelength=lunghezza_onda, displacement=spostamento).show()


## Demo 2 — Frequenza, periodo e lunghezza d’onda

In [ ]:
frequenza = 200.0             # Hz
temperatura = 20.0            # °C
al.space_time_demo(freq=frequenza, temperature=temperatura).show()


## Demo 3 — Suono diretto e riflesso

In [ ]:
distanza_parete = 50.0        # metri
peso_riflessione = 0.5
assert 1 <= distanza_parete <= 100 and 0 <= peso_riflessione <= 0.8
ritardo = 2*distanza_parete/al.SPEED_OF_SOUND_AIR
_, ping = al.decaying_tone(f0=900, tau=1/55, attack=.003, duration=.08, sr=FS)
ping *= .4
y = al.delay_signal(ping, ritardo, gain=peso_riflessione, sr=FS)
al.plot_waveform(y, FS, title=f"Diretto e riflesso: ritardo {ritardo*1000:.1f} ms")
display(al.safe_audio(al.fade_edges(y, FS), FS, normalize=False))


## Demo 4 — Separare gli effetti dei tre parametri

In [ ]:
ampiezza = 1.0
frequenza = 5.0               # Hz, frequenza lenta solo per il grafico
fase = 0.0                   # radianti
al.sinusoid_demo(A=ampiezza, f=frequenza, phi=fase).show()


## Demo 14 — Stessa ampiezza, frequenze diverse

In [ ]:
frequenza_base = 220.0
ampiezza = 0.08
frequenze = tuple(frequenza_base*k for k in (1,2,4))
al.same_amplitude_demo(freqs=frequenze, amp=ampiezza).show()
for f in frequenze:
    print(f"{f:g} Hz — ampiezza {ampiezza:g}")
    display(al.safe_audio(al.fade_edges(al.sine(f,.8,FS,amplitude=ampiezza),FS),FS,normalize=False))


## Demo 4 bis — Stessa ampiezza, frequenze diverse

In [ ]:
frequenze = (220.0, 440.0, 880.0)
ampiezza = 0.08
al.same_amplitude_demo(freqs=frequenze, amp=ampiezza).show()
for f in frequenze:
    print(f"{f:g} Hz — stesso valore di ampiezza")
    display(al.safe_audio(al.fade_edges(al.sine(f,.8,FS,amplitude=ampiezza),FS),FS,normalize=False))


## Demo 10 — La stessa potenza si distribuisce su sfere più grandi

In [ ]:
potenza = 1.0                # W, sorgente puntiforme in campo libero
riferimento = 1.0            # metri
r = np.linspace(riferimento,20*riferimento,400)
intensita = potenza/(4*np.pi*r**2)
i0 = potenza/(4*np.pi*riferimento**2)
al.plot_curves([("Intensità",r,intensita)], x_title="Distanza [m]",y_title="Intensità [W/m²]")
al.plot_curves([("Livello relativo",r,al.db_intensity(intensita,i_ref=i0))],x_title="Distanza [m]",y_title="Differenza [dB]")
for rapporto in (1,2,4,10):
    print(f"Distanza ×{rapporto}: intensità ×{1/rapporto**2:g}; differenza {al.db_intensity(1/rapporto**2,i_ref=1):.1f} dB")


## Demo 11 — Rapporti fisici e differenze in dB

In [ ]:
rapporto_pressione = 2.0
rapporto_intensita = 2.0
rapporti = np.geomspace(.1,10,400)
al.plot_curves([("Pressione",rapporti,al.db_pressure(rapporti,p_ref=1)),
                ("Intensità",rapporti,al.db_intensity(rapporti,i_ref=1))],
               x_title="Rapporto fisico",y_title="Differenza [dB]",x_scale="log")
print("Pressione:",al.db_pressure(rapporto_pressione,p_ref=1),"dB")
print("Intensità:",al.db_intensity(rapporto_intensita,i_ref=1),"dB")


## Demo 12 — Una sorgente, due sorgenti non correlate

In [ ]:
numero_sorgenti = 2
assert 1 <= numero_sorgenti <= 16
sorgenti = [.04*al.white_noise(FS*2,seed=k) for k in range(numero_sorgenti)]
somma = np.sum(sorgenti,axis=0)
potenze = [al.rms(x)**2 for x in sorgenti]
print(f"Previsione: {10*np.log10(numero_sorgenti):.2f} dB")
print(f"Misura: {al.db_intensity(al.rms(somma)**2,i_ref=potenze[0]):.2f} dB")
plt.bar(["Singola","Somma"],[potenze[0],al.rms(somma)**2]); plt.ylabel("Potenza digitale media"); plt.show()
al.compare_audio([("Una sorgente",al.fade_edges(sorgenti[0],FS)),("Somma",al.fade_edges(somma,FS))],FS)


## Demo 5 — Frequenza fisica e altezza percepita

In [ ]:
frequenza_base = 440.0
intervalli = (-12, 0, 12)     # semitoni
frequenze = [frequenza_base*2**(s/12) for s in intervalli]
plt.bar([str(s) for s in intervalli],frequenze); plt.xlabel("Semitoni rispetto alla base"); plt.ylabel("Hz"); plt.show()
for f in frequenze:
    print(f"{f:g} Hz")
    display(al.safe_audio(al.fade_edges(al.sine(f,.8,FS,amplitude=.08),FS),FS,normalize=False))


## Demo 9 — Stessa fondamentale, timbro diverso

In [ ]:
frequenza_base = 440.0
numero_armoniche = 6
rms_comune = .08
puro = al.sine(frequenza_base,1.0,FS)
complesso = al.sum_harmonics(range(1,numero_armoniche+1),f0=frequenza_base,duration=1,sr=FS)
versioni = [(nome,x*rms_comune/al.rms(x)) for nome,x in [("Tono puro",puro),("Somma armonica",complesso)]]
al.plot_signal_rows(versioni,FS,seconds=.02,title="Stessa altezza, forma d'onda diversa")
al.plot_spectra(versioni,FS,up_to=4000)
al.compare_audio([(nome,al.fade_edges(x,FS)) for nome,x in versioni],FS)


## Demo 6 — Interferenza controllata dalla fase

In [ ]:
differenza_fase = 180.0       # gradi
frequenza = 440.0
phi = np.deg2rad(differenza_fase)
al.phase_interference(delta_phi=phi).show()
t = al.time_axis(.8,FS)
x = al.sine(frequenza,.8,FS,amplitude=.08)
z = .08*np.sin(2*np.pi*frequenza*t+phi)
al.compare_audio([("Primo tono",al.fade_edges(x,FS)),("Somma",al.fade_edges(x+z,FS))],FS)


## Demo 7 — Vedere e ascoltare i battimenti

In [ ]:
frequenza_1 = 440.0
frequenza_2 = 443.0
x = al.sum_tones([frequenza_1,frequenza_2],2,FS,amplitude=.16)
al.plot_waveform(x,FS,title=f"Battimenti: {abs(frequenza_2-frequenza_1):g} Hz")
display(al.safe_audio(al.fade_edges(x,FS),FS,normalize=False))


## Demo 13 — Stessa ampiezza digitale, sensazioni diverse

In [ ]:
frequenze = (100.0, 1000.0, 5000.0)
ampiezza = 0.08
versioni = [(f"{f:g} Hz",al.sine(f,.8,FS,amplitude=ampiezza)) for f in frequenze]
for nome,x in versioni:
    print(nome,"— stessa ampiezza",ampiezza)
    display(al.safe_audio(al.fade_edges(x,FS),FS,normalize=False))
plt.bar([n for n,_ in versioni],[al.rms(x) for _,x in versioni]); plt.ylabel("RMS digitale"); plt.ylim(0,.2); plt.show()
print("Il confronto dipende da udito e riproduzione; non misura una curva isofonica.")
